In [ ]:
# ==============================================================
# 03 – CNN Encoder for Alternative Data (Innovative Version)
# RQ2: Does CNN-encoded sequential alternative data improve
#      credit risk prediction over tabular features?
# Production-bank ready + Multi-country support
# ==============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ROOT = Path(".")
DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1. Load Sequential Alternative Data + Labels
# --------------------------------------------------------------
seq_path = DATA_SYNTHETIC / "global_transaction_sequences.npy"
meta_path = DATA_SYNTHETIC / "global_credit_from_german.csv"

if not seq_path.exists() or not meta_path.exists():
    raise FileNotFoundError("Please run the upgraded 01_EDA notebook first.")

txn_seq = np.load(seq_path)                    # shape: (N, 30, 8)
df = pd.read_csv(meta_path)

assert len(txn_seq) == len(df), "Sequence and metadata length mismatch!"

y = df["default"].values.astype(np.int64)
thin = df["thin_file"].values.astype(np.int64)
countries = df["country"].values

print(f"Transaction sequences: {txn_seq.shape}")
print(f"Default rate: {y.mean():.2%}")

# --------------------------------------------------------------
# 2. Train / Validation Split
# --------------------------------------------------------------
idx = np.arange(len(y))
train_idx, val_idx = train_test_split(idx, test_size=0.25, random_state=42, stratify=y)

X_train = torch.tensor(txn_seq[train_idx], dtype=torch.float32)
X_val   = torch.tensor(txn_seq[val_idx],   dtype=torch.float32)
y_train = torch.tensor(y[train_idx], dtype=torch.long)
y_val   = torch.tensor(y[val_idx],   dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val),     batch_size=256, shuffle=False)

print(f"Train sequences: {X_train.shape} | Val: {X_val.shape}")

# --------------------------------------------------------------
# 3. Innovative CNN Encoder Architecture
#    - Multi-scale Temporal Convolutions
#    - Residual connections
#    - Squeeze-and-Excitation style attention
# --------------------------------------------------------------
class ResidualBlock1D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm1d(channels)
        self.act   = nn.ReLU()

    def forward(self, x):
        residual = x
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return self.act(out)


class TemporalAttention(nn.Module):
    """Lightweight channel-wise attention"""
    def __init__(self, channels):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // 4),
            nn.ReLU(),
            nn.Linear(channels // 4, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: [B, C, T]
        w = self.fc(x).unsqueeze(-1)   # [B, C, 1]
        return x * w


class InnovativeCNNEncoder(nn.Module):
    """
    Production-grade sequential encoder for alternative data.
    Input : [B, T=30, C=8]
    Output: embedding [B, emb_dim] + optional classification logit
    """
    def __init__(self, in_channels=8, emb_dim=64, num_classes=2):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )

        self.res1 = ResidualBlock1D(32)
        self.res2 = ResidualBlock1D(32)

        self.down = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU()
        )

        self.res3 = ResidualBlock1D(64)
        self.attn = TemporalAttention(64)

        self.pool = nn.AdaptiveAvgPool1d(1)
        self.embedding = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, emb_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Linear(emb_dim, num_classes)

    def forward(self, x, return_embedding=False):
        # x: [B, T, C] → [B, C, T]
        x = x.permute(0, 2, 1)

        x = self.stem(x)
        x = self.res1(x)
        x = self.res2(x)
        x = self.down(x)
        x = self.res3(x)
        x = self.attn(x)
        x = self.pool(x)

        emb = self.embedding(x)
        logits = self.classifier(emb)

        if return_embedding:
            return emb, logits
        return logits


model = InnovativeCNNEncoder(in_channels=8, emb_dim=64).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# --------------------------------------------------------------
# 4. Training
# --------------------------------------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

EPOCHS = 25
best_auc = 0.0
history = {"train_loss": [], "val_auc": []}

print("\nTraining Innovative CNN Encoder...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    # Validation
    model.eval()
    all_probs, all_true = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs.append(probs)
            all_true.append(yb.numpy())

    val_auc = roc_auc_score(np.concatenate(all_true), np.concatenate(all_probs))
    history["train_loss"].append(epoch_loss / len(train_loader))
    history["val_auc"].append(val_auc)

    scheduler.step(val_auc)

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), RESULTS / "cnn_encoder_best.pt")

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/{EPOCHS} | Loss: {history['train_loss'][-1]:.4f} | Val AUC: {val_auc:.4f}")

print(f"\nBest Validation AUC: {best_auc:.4f}")

# --------------------------------------------------------------
# 5. Extract Embeddings for downstream MARL (Production use)
# --------------------------------------------------------------
model.load_state_dict(torch.load(RESULTS / "cnn_encoder_best.pt", map_location=device))
model.eval()

def extract_embeddings(sequences):
    """Production embedding extraction function"""
    model.eval()
    loader = DataLoader(torch.tensor(sequences, dtype=torch.float32), batch_size=256)
    embs = []
    with torch.no_grad():
        for xb in loader:
            xb = xb.to(device)
            emb, _ = model(xb, return_embedding=True)
            embs.append(emb.cpu().numpy())
    return np.concatenate(embs, axis=0)

# Full embeddings
all_embeddings = extract_embeddings(txn_seq)
print(f"Extracted embeddings shape: {all_embeddings.shape}")

np.save(DATA_PROCESSED / "cnn_embeddings.npy", all_embeddings)
np.save(DATA_PROCESSED / "cnn_labels.npy", y)
np.save(DATA_PROCESSED / "cnn_thin.npy", thin)

print("✓ CNN embeddings saved to data/processed/cnn_embeddings.npy")

# --------------------------------------------------------------
# 6. Quick Visualization
# --------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["val_auc"], label="Val ROC-AUC", color="green")
axes[1].set_title("Validation ROC-AUC")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS / "cnn_training_curves.png", dpi=140, bbox_inches="tight")
plt.show()

print("\n✅ Innovative CNN Encoder completed.")
print("Embeddings are ready for Feature Fusion and Multi-Agent Training (RQ1 + RQ2).")